# **🧾 Incertidumbre, política de tres zonas y contingencia**

Tres preguntas que el resultado del modelo final deja abiertas:

1. **¿Cuánta incertidumbre tienen las cifras principales?** El 78,9% de la ganancia máxima, los +232 mil sobre aprobar todo y la mejora del tuning se reportaron como números puntuales. Aquí se les pone un intervalo de confianza por bootstrap.
2. **¿Cuánto vale una tercera zona?** Cerca del umbral, el modelo no discrimina bien. En lugar de rechazar, se puede pedir autenticación (3DS): frena parte del fraude, pero una parte de los clientes legítimos abandona. Se simula esa política con supuestos explícitos de abandono y efectividad.
3. **¿Qué hacer si falta el `score`?** Es la variable más valiosa y depende de otro sistema. Se mide un modelo de contingencia sin ella, frente a las alternativas sin modelo.

Todo se evalúa con las predicciones out-of-fold del pipeline productivo, con la misma partición del notebook 04 (semilla 7), y la tercera zona y la contingencia se verifican además en la última semana, prediciendo solo con el pasado.

## **📚 Librerías**

In [1]:
# Librerías para manipulación de datos
import pandas as pd
import numpy as np

# Mostrar varias tablas en una misma celda
from IPython.display import display

# Métrica de ordenamiento
from sklearn.metrics import roc_auc_score

# Módulos del proyecto: datos, pipeline productivo y función de ganancia
from fraude_shipping.features import load_data, make_folds
from fraude_shipping.production.pipeline import THRESHOLD, FraudPipeline
from fraude_shipping.profit import PROFIT_MARGIN, compute_profit

In [2]:
# Misma partición de la CV aleatoria que validó el modelo final en el notebook 04
SEMILLA_VALIDACION = 7
REMUESTRAS_BOOTSTRAP = 1000
SEMILLA_BOOTSTRAP = 42
# Última semana: se predice entrenando solo con el pasado
SEMANA_EVALUACION = '2020-04-15'

# Tercera zona (3DS): proporción de legítimas que abandonan y de fraudes que la autenticación frena (supuestos)
ABANDONOS = [0.05, 0.10, 0.20]
FRAUDES_FRENADOS = [0.70, 0.85, 0.95]
ESCENARIO_CENTRAL = {'abandono': 0.10, 'fraude_frenado': 0.85}
GRILLA_UMBRALES = np.round(np.arange(0.01, 0.61, 0.01), 2)
# Tope de transacciones desafiadas: la fricción y el costo operativo de 3DS no escalan sin límite (supuesto)
DESAFIO_MAXIMO = 0.05
TOPES_DESAFIO = [0.02, 0.05, 0.10, 1.0]

# Contingencia: rechazar las transacciones por encima de estos percentiles de monto
PERCENTILES_MONTO = [0.80, 0.90, 0.95]

In [3]:
# Ganancia en % de la máxima e intervalos por bootstrap


def ganancia_pct_maxima(fraude, monto, aprobada):
    """Ganancia de una decisión en % de la ganancia máxima."""
    return compute_profit(fraude, monto, aprobada) / compute_profit(fraude, monto, fraude == 0) * 100


def intervalo_bootstrap(estadistico, cantidad_filas):
    """Media e IC95 de un estadístico calculado sobre remuestras bootstrap de las filas."""
    generador = np.random.default_rng(SEMILLA_BOOTSTRAP)
    valores = np.array([estadistico(generador.integers(0, cantidad_filas, cantidad_filas)) for _ in range(REMUESTRAS_BOOTSTRAP)])
    return pd.Series({'media': valores.mean(), 'ic95_inferior': np.percentile(valores, 2.5), 'ic95_superior': np.percentile(valores, 97.5)})

## **🔢 Datos y predicciones out-of-fold**

In [4]:
datos = load_data('../data/raw/dataset.csv')
fraude, monto = datos['fraude'].to_numpy(), datos['monto'].to_numpy()
folds = make_folds(datos['fraude'], seed=SEMILLA_VALIDACION)


def prediccion_out_of_fold(datos_modelo, parametros=None):
    """Probabilidad out-of-fold del pipeline productivo con los parámetros dados (None: los de producción)."""
    probabilidad = np.zeros(len(datos_modelo))
    for indices_train, indices_validacion in folds:
        pipeline = FraudPipeline(params=parametros).fit(datos_modelo.iloc[indices_train])
        probabilidad[indices_validacion] = pipeline.predict_proba(datos_modelo.iloc[indices_validacion])
    return probabilidad


# Modelo final, LightGBM con sus parámetros por defecto y modelo sin score (la variable queda nula al entrenar y al predecir)
datos_sin_score = datos.assign(score=np.nan)
probabilidades = {
    'final': prediccion_out_of_fold(datos),
    'lightgbm_defecto': prediccion_out_of_fold(datos, parametros={}),
    'sin_score': prediccion_out_of_fold(datos_sin_score),
}
pd.DataFrame({
    nombre: {'ganancia_pct_maxima': ganancia_pct_maxima(fraude, monto, probabilidad < THRESHOLD), 'auc_roc': roc_auc_score(fraude, probabilidad)}
    for nombre, probabilidad in probabilidades.items()
}).T.round(3)

,ganancia_pct_maxima,auc_roc
final,78.894,0.889
lightgbm_defecto,78.462,0.889
sin_score,76.997,0.853


**💡 Hallazgos**

- El pipeline reproduce el modelo final del notebook 04: **78,9%** de la ganancia máxima y AUC de 0,889 con el umbral de 0,20.
- Con los parámetros por defecto de LightGBM, el mismo pipeline da 78,5%, con el mismo AUC.
- **Sin `score`, el modelo da 77,0%** y el AUC baja de 0,889 a 0,853: pierde menos de 2 puntos.

In [5]:
# Última semana predicha solo con el pasado, con el modelo completo y con el modelo sin score
pasado = (datos['fecha'] < SEMANA_EVALUACION).to_numpy()
ultima_semana = ~pasado
datos_pasado, datos_semana = datos[pasado], datos[ultima_semana]
fraude_semana, monto_semana = fraude[ultima_semana], monto[ultima_semana]
probabilidad_completo_semana = FraudPipeline().fit(datos_pasado).predict_proba(datos_semana)
probabilidad_sin_score_semana = FraudPipeline().fit(datos_pasado.assign(score=np.nan)).predict_proba(datos_semana.assign(score=np.nan))

## **📏 Intervalos de las cifras principales**

In [6]:
# Ganancia del modelo final, diferencia en dinero frente a aprobar todo y aporte del tuning, con IC95 por bootstrap
aprobar_todo = np.ones(len(datos), dtype=bool)
aprobada_final = probabilidades['final'] < THRESHOLD
aprobada_defecto = probabilidades['lightgbm_defecto'] < THRESHOLD

estadisticos = {
    'ganancia_final_pct': lambda indices: ganancia_pct_maxima(fraude[indices], monto[indices], aprobada_final[indices]),
    'ganancia_aprobar_todo_pct': lambda indices: ganancia_pct_maxima(fraude[indices], monto[indices], aprobar_todo[indices]),
    'mejora_vs_aprobar_todo_mil': lambda indices: (
        compute_profit(fraude[indices], monto[indices], aprobada_final[indices])
        - compute_profit(fraude[indices], monto[indices], aprobar_todo[indices])) / 1000,
    'mejora_tuning_puntos': lambda indices: (
        ganancia_pct_maxima(fraude[indices], monto[indices], aprobada_final[indices])
        - ganancia_pct_maxima(fraude[indices], monto[indices], aprobada_defecto[indices])),
}
pd.DataFrame({nombre: intervalo_bootstrap(estadistico, len(datos)) for nombre, estadistico in estadisticos.items()}).T.round(2)

,media,ic95_inferior,ic95_superior
ganancia_final_pct,78.91,77.56,80.08
ganancia_aprobar_todo_pct,63.36,61.25,65.49
mejora_vs_aprobar_todo_mil,232.46,207.09,258.31
mejora_tuning_puntos,0.42,-0.51,1.28


**💡 Hallazgos**

- **Modelo final: 78,9% de la ganancia máxima, IC95 de 77,6% a 80,1%.** Aprobar todo queda en 63,4% (IC95 de 61,3% a 65,5%): los intervalos no se tocan.
- **Frente a aprobar todo, +232 mil, IC95 de 207 a 258 mil** sobre las 150.000 transacciones.
- **El tuning aporta +0,4 puntos, IC95 de −0,5 a +1,3: no se distingue de cero.** Con el umbral de la política y las 19 variables finales, la mejora del tuning es compatible con ruido.

In [7]:
# Las mismas cifras fold a fold: la variación entre folds de 30.000 transacciones
filas = []
for numero, (_, indices) in enumerate(folds, start=1):
    filas.append({
        'fold': numero,
        'ganancia_final_pct': ganancia_pct_maxima(fraude[indices], monto[indices], aprobada_final[indices]),
        'mejora_tuning_puntos': (ganancia_pct_maxima(fraude[indices], monto[indices], aprobada_final[indices])
                                 - ganancia_pct_maxima(fraude[indices], monto[indices], aprobada_defecto[indices])),
    })
por_fold = pd.DataFrame(filas).set_index('fold')
display(por_fold.round(2))
por_fold.agg(['mean', 'std']).round(2)

,ganancia_final_pct,mejora_tuning_puntos
fold,,
1,78.86,1.45
2,78.06,1.28
3,81.11,0.03
4,78.22,-0.29
5,78.23,-0.34


,ganancia_final_pct,mejora_tuning_puntos
mean,78.90,0.42
std,1.27,0.87


**💡 Hallazgos**

- Entre folds, la ganancia va de 78,1% a 81,1% (desvío 1,3), en línea con el intervalo del bootstrap.
- El tuning mejora en 3 de los 5 folds (+1,5, +1,3 y 0,0) y empeora en 2 (−0,3 y −0,3). En el notebook 04, con `perfil_onp` y con el umbral óptimo de cada modelo, mejoraba en 4 de 5: en el modelo final, su aporte es todavía más incierto.

## **🚦 Política de tres zonas**

Con dos umbrales, cada transacción se **aprueba** (p < t₁), se **desafía con 3DS** (t₁ ≤ p < t₂) o se **rechaza** (p ≥ t₂). En la zona intermedia, una legítima deja su margen solo si no abandona (`1 − abandono`), y un fraude se pierde solo si pasa la autenticación (`1 − fraude_frenado`). Con t₁ = t₂ se recupera la política de dos zonas. Como el modelo no cobra la fricción de las legítimas que sí completan la autenticación ni el costo operativo de 3DS, sin límite desafiaría a una parte grande del tráfico: por eso se fija un **tope de transacciones desafiadas** (5% en los escenarios, y de 2% a sin límite en el escenario central).

Los dos parámetros de 3DS son **supuestos**: se muestra una grilla de escenarios. Los umbrales se eligen con las predicciones out-of-fold previas a la última semana y se evalúan en ella, con el modelo entrenado solo con el pasado, para no elegir y medir con las mismas filas.

In [8]:
# Ganancia de la política de tres zonas y búsqueda de los dos umbrales en una grilla


def ganancia_tres_zonas(fraude_zona, monto_zona, probabilidad, umbral_desafio, umbral_rechazo, abandono, fraude_frenado):
    """Ganancia en % de la máxima: aprobar bajo el primer umbral, desafiar entre ambos y rechazar desde el segundo."""
    legitima = fraude_zona == 0
    aprobada = probabilidad < umbral_desafio
    desafiada = (probabilidad >= umbral_desafio) & (probabilidad < umbral_rechazo)
    ganancia = (
        PROFIT_MARGIN * monto_zona[aprobada & legitima].sum() - monto_zona[aprobada & ~legitima].sum()
        + PROFIT_MARGIN * (1 - abandono) * monto_zona[desafiada & legitima].sum()
        - (1 - fraude_frenado) * monto_zona[desafiada & ~legitima].sum()
    )
    return ganancia / (PROFIT_MARGIN * monto_zona[legitima].sum()) * 100


def mejores_umbrales(fraude_zona, monto_zona, probabilidad, abandono, fraude_frenado, desafio_maximo=DESAFIO_MAXIMO):
    """Par de umbrales de la grilla (desafío ≤ rechazo) con mayor ganancia, sin desafiar más que el tope."""
    candidatos = [
        (ganancia_tres_zonas(fraude_zona, monto_zona, probabilidad, desafio, rechazo, abandono, fraude_frenado), desafio, rechazo)
        for desafio in GRILLA_UMBRALES for rechazo in GRILLA_UMBRALES
        if rechazo >= desafio and ((probabilidad >= desafio) & (probabilidad < rechazo)).mean() <= desafio_maximo
    ]
    return max(candidatos)

In [9]:
# Escenarios con el tope de 5%: umbrales elegidos con las predicciones out-of-fold previas a la última semana, evaluados en ella
probabilidad_pasado = probabilidades['final'][pasado]
argumentos_semana = (fraude_semana, monto_semana, probabilidad_completo_semana)

filas = []
for abandono in ABANDONOS:
    for fraude_frenado in FRAUDES_FRENADOS:
        _, desafio, rechazo = mejores_umbrales(fraude[pasado], monto[pasado], probabilidad_pasado, abandono, fraude_frenado)
        tres_zonas = ganancia_tres_zonas(*argumentos_semana, desafio, rechazo, abandono, fraude_frenado)
        dos_zonas = ganancia_tres_zonas(*argumentos_semana, THRESHOLD, THRESHOLD, abandono, fraude_frenado)
        desafiadas = (probabilidad_completo_semana >= desafio) & (probabilidad_completo_semana < rechazo)
        filas.append({
            'abandono': abandono, 'fraude_frenado': fraude_frenado, 'umbral_desafio': desafio, 'umbral_rechazo': rechazo,
            'desafiadas_pct': desafiadas.mean() * 100, 'ganancia_tres_zonas': tres_zonas,
            'ganancia_dos_zonas': dos_zonas, 'diferencia': tres_zonas - dos_zonas,
        })

escenarios = pd.DataFrame(filas).set_index(['abandono', 'fraude_frenado'])
escenarios.round(2)

umbral_desafio  umbral_rechazo  desafiadas_pct  \
abandono fraude_frenado                                                   
0.05     0.70                      0.14            0.34            4.96   
         0.85                      0.15            0.40            4.77   
         0.95                      0.16            0.47            4.82   
0.10     0.70                      0.14            0.34            4.96   
         0.85                      0.15            0.40            4.77   
         0.95                      0.16            0.47            4.82   
0.20     0.70                      0.11            0.20            4.88   
         0.85                      0.15            0.40            4.77   
         0.95                      0.16            0.47            4.82   

                         ganancia_tres_zonas  ganancia_dos_zonas  diferencia  
abandono fraude_frenado                                                       
0.05     0.70                          83.98               79.46        4.52  
         0.85                          85.38               79.46        5.92  
         0.95                          86.45               79.46        6.99  
0.10     0.70                          83.65               79.46        4.19  
         0.85                          85.05               79.46        5.59  
         0.95                          86.11               79.46        6.65  
0.20     0.70                          82.10               79.46        2.64  
         0.85                          84.39               79.46        4.93  
         0.95                          85.43               79.46        5.97

**💡 Hallazgos**

- **Con 5% de transacciones desafiadas, la tercera zona gana entre +2,6 y +7,0 puntos** en la última semana, según el supuesto de abandono y de fraude frenado. Los umbrales elegidos con el pasado desafían entre ~0,15 y ~0,40 y rechazan por encima.
- La ganancia viene de dos lados: las transacciones entre 0,20 y 0,40, que antes se rechazaban, recuperan el margen de las legítimas que completan la autenticación; y las de 0,15 a 0,20, que antes se aprobaban, frenan parte del fraude.
- El resultado depende sobre todo de cuánto fraude frena 3DS: con 70%, entre +2,6 y +4,5; con 95%, entre +6,0 y +7,0.

In [10]:
# Escenario central con distintos topes de transacciones desafiadas
filas = []
for tope in TOPES_DESAFIO:
    _, desafio, rechazo = mejores_umbrales(fraude[pasado], monto[pasado], probabilidad_pasado, **ESCENARIO_CENTRAL, desafio_maximo=tope)
    desafiadas = (probabilidad_completo_semana >= desafio) & (probabilidad_completo_semana < rechazo)
    tres_zonas = ganancia_tres_zonas(*argumentos_semana, desafio, rechazo, **ESCENARIO_CENTRAL)
    filas.append({
        'tope_desafiadas': tope, 'umbral_desafio': desafio, 'umbral_rechazo': rechazo, 'desafiadas_pct': desafiadas.mean() * 100,
        'ganancia_tres_zonas': tres_zonas,
        'diferencia': tres_zonas - ganancia_tres_zonas(*argumentos_semana, THRESHOLD, THRESHOLD, **ESCENARIO_CENTRAL),
    })

pd.DataFrame(filas).set_index('tope_desafiadas').round(2)

,umbral_desafio,umbral_rechazo,desafiadas_pct,ganancia_tres_zonas,diferencia
tope_desafiadas,,,,,
0.02,0.17,0.23,1.80,82.22,2.76
0.05,0.15,0.40,4.77,85.05,5.59
0.10,0.09,0.41,9.54,87.24,7.78
1.00,0.02,0.58,32.09,89.13,9.67


**💡 Hallazgos**

- **Cuantas más transacciones se desafían, más se gana**: +2,8 puntos con 2%, +5,6 con 5% y +7,8 con 10%. Sin tope, se desafiaría el 32% del tráfico, algo inviable: el modelo no cobra la fricción de las legítimas que completan la autenticación ni el costo operativo de 3DS.
- El tope es, por lo tanto, una decisión de negocio: cuánta fricción se acepta a cambio de ganancia.

In [11]:
# Escenario central con el tope de 5%: IC95 de la diferencia entre tres y dos zonas en la última semana
_, desafio_central, rechazo_central = mejores_umbrales(
    fraude[pasado], monto[pasado], probabilidad_pasado, ESCENARIO_CENTRAL['abandono'], ESCENARIO_CENTRAL['fraude_frenado'])


def diferencia_tres_zonas(indices):
    """Diferencia entre tres y dos zonas en una remuestra de la última semana, con el escenario central."""
    argumentos = (fraude_semana[indices], monto_semana[indices], probabilidad_completo_semana[indices])
    return (ganancia_tres_zonas(*argumentos, desafio_central, rechazo_central, **ESCENARIO_CENTRAL)
            - ganancia_tres_zonas(*argumentos, THRESHOLD, THRESHOLD, **ESCENARIO_CENTRAL))


print(f'Escenario central: desafiar entre {desafio_central:.2f} y {rechazo_central:.2f}')
intervalo_bootstrap(diferencia_tres_zonas, ultima_semana.sum()).round(2)

Escenario central: desafiar entre 0.15 y 0.40


media            5.61
ic95_inferior    4.10
ic95_superior    7.38
dtype: float64

**💡 Hallazgos**

- En el escenario central (10% de abandono y 85% de fraude frenado), la tercera zona gana **+5,6 puntos en la última semana, IC95 de +4,1 a +7,4**. Con estos supuestos, la mejora es clara; la incertidumbre principal está en los supuestos, no en los datos.

## **🛟 Contingencia sin `score`**

In [12]:
# Alternativas si falta el score: aprobar todo, rechazar por monto, modelo sin score y, como referencia, el modelo completo


def comparar_contingencias(fraude_zona, monto_zona, probabilidad_completo, probabilidad_sin_score):
    """Ganancia en % de la máxima de cada alternativa sobre las mismas transacciones."""
    filas = {'aprobar_todo': ganancia_pct_maxima(fraude_zona, monto_zona, np.ones(len(fraude_zona), dtype=bool))}
    for percentil in PERCENTILES_MONTO:
        limite = np.quantile(monto, percentil)
        filas[f'rechazar_monto_sobre_p{int(percentil * 100)}'] = ganancia_pct_maxima(fraude_zona, monto_zona, monto_zona < limite)
    filas['modelo_sin_score'] = ganancia_pct_maxima(fraude_zona, monto_zona, probabilidad_sin_score < THRESHOLD)
    filas['modelo_completo'] = ganancia_pct_maxima(fraude_zona, monto_zona, probabilidad_completo < THRESHOLD)
    return pd.Series(filas)


pd.DataFrame({
    'cv_aleatoria': comparar_contingencias(fraude, monto, probabilidades['final'], probabilidades['sin_score']),
    'ultima_semana': comparar_contingencias(fraude_semana, monto_semana, probabilidad_completo_semana, probabilidad_sin_score_semana),
}).round(2)

,cv_aleatoria,ultima_semana
aprobar_todo,63.40,67.14
rechazar_monto_sobre_p80,28.03,29.05
rechazar_monto_sobre_p90,39.40,40.83
rechazar_monto_sobre_p95,48.79,49.69
modelo_sin_score,77.00,76.97
modelo_completo,78.89,79.46


**💡 Hallazgos**

- **Sin `score`, el modelo de contingencia da 77,0%** en la CV y en la última semana: pierde 1,9 y 2,5 puntos frente al modelo completo, y queda muy por encima de aprobar todo (63,4% y 67,1%).
- **Rechazar por monto es mucho peor que aprobar todo**: entre 28% y 50%, según el percentil. Aunque el 20% más caro concentre la mayor parte de la pérdida por fraude, también concentra la mayor parte del margen de las legítimas.
- Como respaldo, el orden es: modelo sin `score` si falta la variable, y aprobar todo si falla el modelo. Un límite de monto no sirve como respaldo.

## **🏁 Conclusión**

- **Las cifras principales tienen intervalos estrechos**: 78,9% (IC95 de 77,6% a 80,1%) y +232 mil frente a aprobar todo (IC95 de 207 a 258 mil). **El aporte del tuning no**: +0,4 puntos, con un intervalo que incluye el cero.
- **Una tercera zona con 3DS es la mejora más grande disponible**: con 5% de las transacciones desafiadas entre ~0,15 y ~0,40, gana entre +2,6 y +7,0 puntos en la última semana según los supuestos, y +5,6 en el escenario central. Antes de adoptarla hay que medir en producción cuánto fraude frena 3DS y cuántos clientes abandonan.
- **El modelo sin `score` es una contingencia real**: 77,0%, menos de 2,5 puntos debajo del modelo completo. Si falla el modelo, el respaldo es aprobar todo; rechazar por monto destruye margen.